In [ ]:
# from delta import *
# from pyspark.sql import SparkSession

# builder = SparkSession.builder \
#     .appName("RetailPipeline") \
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [1]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip

    builder = SparkSession.builder \
        .appName("retail_pipeline") \
        .master("local[*]") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

ERROR StatusLogger Reconfiguration failed: No configuration found for '5c29bfd' at 'null' in 'null'
ERROR StatusLogger Reconfiguration failed: No configuration found for 'Default' at 'null' in 'null'
26/04/23 16:25:01 WARN Utils: Your hostname, YTuSurface5 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 16:25:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/yiqingtu/spark/spark-3.5.5-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/yiqingtu/.ivy2/cache
The jars for the packages stored in: /home/yiqingtu/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8f9c44f2-1f8b-4fae-abbe-191c23be213c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 190ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |

In [30]:
bronze_orders_path = "../delta/bronze/orders"
df_bronze = spark.read.load(bronze_orders_path)

# need panda library
df_bronze.limit(1000).toPandas()


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,subcategory,product_name,sales,quantity,discount,profit
0,764,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-EN-10001990,Office Supplies,Envelopes,Staple envelope,11.36,2.0,0.0,5.3392
1,765,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-EN-10001532,Office Supplies,Envelopes,Brown Kraft Recycled Envelopes,50.94,3.0,0.0,25.4700
2,766,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,TEC-AC-10003174,Technology,Accessories,Plantronics S12 Corded Telephone Headset System,646.74,6.0,0.0,258.6960
3,767,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-BI-10004187,Office Supplies,Binders,3-ring staple pack,5.64,3.0,0.0,2.7072
4,768,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-ST-10000025,Office Supplies,Storage,Fellowes Stor/Drawer Steel Plus Storage Drawers,572.58,6.0,0.0,34.3548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,6475,CA-2014-149524,2014-01-14,2014-01-15,First Class,BS-11590,Brendan Sweed,Corporate,United States,Philadelphia,...,19140,East,FUR-BO-10003433,Furniture,Bookcases,Sauder Cornerstone Collection Library,61.96,4.0,0.5,-53.2856
95,6475,CA-2014-149524,2014-01-14,2014-01-15,First Class,BS-11590,Brendan Sweed,Corporate,United States,Philadelphia,...,19140,East,FUR-BO-10003433,Furniture,Bookcases,Sauder Cornerstone Collection Library,61.96,4.0,0.5,-53.2856
96,717,CA-2014-130092,2014-01-11,2014-01-14,First Class,SV-20365,Seth Vernon,Consumer,United States,Dover,...,19901,East,FUR-FU-10000010,Furniture,Furnishings,"DAX Value U-Channel Document Frames, Easel Back",9.94,2.0,0.0,3.0814
97,717,CA-2014-130092,2014-01-11,2014-01-14,First Class,SV-20365,Seth Vernon,Consumer,United States,Dover,...,19901,East,FUR-FU-10000010,Furniture,Furnishings,"DAX Value U-Channel Document Frames, Easel Back",9.94,2.0,0.0,3.0814


In [3]:
df_bronze.printSchema()

root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)



In [6]:
from pyspark.sql.functions import col
df_bronze = df_bronze \
    .withColumn("sales", col("sales").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int"))

In [8]:
df_bronze.filter(
    col("sales").isNull() | col("profit").isNull()
).show()

+------+--------------+----------+----------+--------------+-----------+-------------+--------+-------------+-------+-----+-----------+-------+---------------+---------------+-----------+--------------------+-----+--------+--------+------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|customer_name| segment|      country|   city|state|postal_code| region|     product_id|       category|subcategory|        product_name|sales|quantity|discount|profit|
+------+--------------+----------+----------+--------------+-----------+-------------+--------+-------------+-------+-----+-----------+-------+---------------+---------------+-----------+--------------------+-----+--------+--------+------+
|  7981|CA-2014-103800|2014-01-03|2014-01-07|Standard Class|   DP-13000|Darren Powers|Consumer|United States|Houston|Texas|      77095|Central|OFF-PA-10000174|Office Supplies|      Paper|"Message Book, Wi...| NULL|      16|     2.0|   0.2|
|  7981|CA-2014-103800|2014-01-03|2014-0

In [10]:
# how many null

df_bronze.filter(col("sales").isNull()).count()

3

In [13]:
# drop null and repeat records

# For business: customer_id is required for downstream customer analytics
df_bronze = df_bronze.filter(col("sales").isNotNull())


In [14]:
# step 2 remove duplicate

from pyspark.sql.functions import count

df_bronze.groupBy("order_id") \
    .count() \
    .filter("count > 1") \
    .show()


+--------------+-----+
|      order_id|count|
+--------------+-----+
|CA-2014-118192|    6|
|CA-2014-109232|    3|
|CA-2014-162775|   15|
|CA-2014-157147|    9|
|CA-2014-130813|    3|
|CA-2014-106054|    3|
|CA-2014-167199|   21|
|CA-2014-105417|    6|
|CA-2014-112326|    9|
|CA-2014-149020|    6|
|CA-2014-135405|    6|
|CA-2014-141817|    3|
|CA-2014-149524|    3|
|CA-2014-130092|    3|
+--------------+-----+



In [15]:
df_bronze.count() - df_bronze.dropDuplicates().count()

64

In [16]:
total = df_bronze.count()
distinct = df_bronze.distinct().count()

print("Duplicate rows:", total - distinct)

Duplicate rows: 64


In [20]:
df_bronze.groupBy(df_bronze.columns) \
    .count() \
    .filter("count > 1") \
    .show(100)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------------+-----------+--------------------+-------+--------+--------+--------+-----+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|    customer_name|    segment|      country|          city|         state|postal_code| region|     product_id|       category|subcategory|        product_name|  sales|quantity|discount|  profit|count|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------------+-----------+--------------------+-------+--------+--------+--------+-----+
|  9629|CA-2014-118192|2014-01-13|2014-01-18|Standard Class|   MM-17920|    Michael Moore|   Consumer|United States|        Newark|          Ohio|      43055|   East|OFF-PA-10002947

In [21]:
# produce report
row_dupes_df = df_bronze.groupBy(df_bronze.columns) \
    .count() \
    .filter(col("count") > 1)


In [22]:
row_duplicates = df_bronze.join(
    row_dupes_df.drop("count"),
    on=df_bronze.columns,
    how="inner"
)

In [23]:
row_duplicates.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("row_level_duplicates")

26/04/23 17:00:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/23 17:00:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/23 17:00:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [25]:
row_duplicates.count()

96

In [28]:
row_duplicates.limit(10).toPandas()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,subcategory,product_name,sales,quantity,discount,profit
0,764,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-EN-10001990,Office Supplies,Envelopes,Staple envelope,11.360,2,0.00,5.3392
1,765,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-EN-10001532,Office Supplies,Envelopes,Brown Kraft Recycled Envelopes,50.940,3,0.00,25.4700
2,766,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,TEC-AC-10003174,Technology,Accessories,Plantronics S12 Corded Telephone Headset System,646.740,6,0.00,258.6960
3,767,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-BI-10004187,Office Supplies,Binders,3-ring staple pack,5.640,3,0.00,2.7072
4,768,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-ST-10000025,Office Supplies,Storage,Fellowes Stor/Drawer Steel Plus Storage Drawers,572.580,6,0.00,34.3548
5,2979,CA-2014-109232,2014-01-13,2014-01-16,Second Class,ND-18370,Natalie DeCherney,Consumer,United States,Mount Pleasant,...,29464,South,FUR-CH-10000422,Furniture,Chairs,Global Highback Leather Tilter in Burgundy,545.940,6,0.00,87.3504
6,4938,CA-2014-157147,2014-01-13,2014-01-18,Standard Class,BD-11605,Brian Dahlen,Consumer,United States,San Francisco,...,94109,West,OFF-ST-10000078,Office Supplies,Storage,Tennsco 6- and 18-Compartment Lockers,1325.850,5,0.00,238.6530
7,4939,CA-2014-157147,2014-01-13,2014-01-18,Standard Class,BD-11605,Brian Dahlen,Consumer,United States,San Francisco,...,94109,West,FUR-BO-10003034,Furniture,Bookcases,"O'Sullivan Elevations Bookcase, Cherry Finish",333.999,3,0.15,3.9294
8,4940,CA-2014-157147,2014-01-13,2014-01-18,Standard Class,BD-11605,Brian Dahlen,Consumer,United States,San Francisco,...,94109,West,OFF-AR-10003514,Office Supplies,Art,4009 Highlighters by Sanford,19.900,5,0.00,6.5670
9,9629,CA-2014-118192,2014-01-13,2014-01-18,Standard Class,MM-17920,Michael Moore,Consumer,United States,Newark,...,43055,East,OFF-PA-10002947,Office Supplies,Paper,Xerox 1923,37.408,7,0.20,13.0928


In [ ]:
# done extract duplicate

# ttd
#    extract key level
#    clean up 


# add structure in markdown


# section 1: read
# section: clean data
# section: deduplicate

# section: check business rule
# section: write silver



In [31]:
# Business check: sales must be >= 0 because refunds are stored separately
df_bronze.filter("sales <= 0").count()

0

26/04/24 12:42:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 24666146 ms exceeds timeout 120000 ms
26/04/24 12:42:54 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/24 12:49:50 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint